In [2]:
from faker import Faker
import pandas as pd
import numpy as np
import kagglehub
import os
import random

c:\Users\janva\DataEngineeringRoadMap\DataGeneration\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
fake = Faker('pl_PL') # Polsa lokalizacja

path = kagglehub.dataset_download("vipullrathod/fish-market")
print("Path to dataset files:", path)
print(os.listdir(path))

csv_path = os.path.join(path, "Fish.csv")
df = pd.read_csv(csv_path)

dfBream = df[df['Species'] == 'Bream']
dfBream_mW, dfBream_stdW = dfBream['Weight'].mean(), dfBream['Weight'].std()
dfBream_mH, dfBream_stdH = dfBream['Height'].mean(), dfBream['Height'].std()

Path to dataset files: C:\Users\janva\.cache\kagglehub\datasets\vipullrathod\fish-market\versions\1
['Fish.csv']


In [4]:
def generate_bream_data(n_samples=50):
    weights = np.random.normal(dfBream_mW, dfBream_stdW, n_samples)
    weights = np.clip(weights, 242, 1100)

    data = []

    for i in range(n_samples):
        if random.random() < 0.05: # 5% szans na brak rekordu (symulacja złych danych)
            id_ryby = ""
        else:
            id_ryby = f"BRM-{fake.unique.random_int(1000, 9999)}"
        record = {
            "ID_Ryby": id_ryby,
            "Data_Polowu": fake.date_between(start_date='-1y', end_date='today'),
            "Rybak": fake.name(),
            "Lokalizacja": fake.city(),
            "Waga_g": round(weights[i], 2),
            "Stan_Zdrowia": fake.random_element(elements=('Zdrowa', 'Zdrowa', 'Zdrowa', 'Chora')), # 75% szans na zdrową
            "Komentarz": fake.sentence(nb_words=5)
        }
        data.append(record)

    return pd.DataFrame(data)

In [11]:
def introduce_chaos(df, corruption_level=0.1):
    dfCorrupted = df.copy()
    nRows = len(dfCorrupted)
    nCorrupt = int(nRows * corruption_level)

    nullIndices = np.random.choice(dfCorrupted.index, nCorrupt // 2, replace=False)
    dfCorrupted.loc[nullIndices, 'Waga_g'] = np.nan

    outlierIndices = np.random.choice(dfCorrupted.index, nCorrupt // 2, replace=False)
    dfCorrupted.loc[outlierIndices, 'Waga_g'] = 500000.0

    return dfCorrupted

In [10]:
dfCleanBream = generate_bream_data(50)
#print(dfCleanBream)

dfCleanBream.to_json("fake_bream_data.json", orient="records", lines=True)
print(dfCleanBream.info())

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID_Ryby       50 non-null     str    
 1   Data_Polowu   50 non-null     object 
 2   Rybak         50 non-null     str    
 3   Lokalizacja   50 non-null     str    
 4   Waga_g        50 non-null     float64
 5   Stan_Zdrowia  50 non-null     str    
 6   Komentarz     50 non-null     str    
dtypes: float64(1), object(1), str(5)
memory usage: 6.5+ KB
None


In [ ]:
dfClean = generate_bream_data(100)

dfFinal = introduce_chaos(dfClean, corruption_level=0.1)

dfFinal.to_json("fish_data.jsonl", orient="records", lines=True, force_ascii=False)

print(dfFinal.info())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID_Ryby       100 non-null    str    
 1   Data_Polowu   100 non-null    object 
 2   Rybak         100 non-null    str    
 3   Lokalizacja   100 non-null    str    
 4   Waga_g        96 non-null     float64
 5   Stan_Zdrowia  100 non-null    str    
 6   Komentarz     100 non-null    str    
dtypes: float64(1), object(1), str(5)
memory usage: 13.0+ KB
None
